# 04 — Frozen held-out evaluation / benchmark / demo

Prepared template; run only after validation selection is frozen. No test-driven tuning.


## Environment and assets

Enable a Kaggle GPU accelerator. Clone/mount this repository, then install its
`traffic-mot/requirements.txt` and `requirements-dev.txt` into an environment with
matching PyTorch/torchvision CUDA wheels. Inspect `pip check` before proceeding.
Clone the official RT-DETR repository at commit
`29320b6fd828f8e0987a71426cf2d961b09dfed7` into `RT-DETR/` if it is absent;
keep this checkout clean. Do not substitute a YOLO detector or auto-download weights.

Mount the real bundle (`best.pth`, `demo_config.yml`, `label_map.json`,
`classes.txt`, `bundle_manifest.json`). Edit paths below; REPLACE_ME must not remain.
Preflight rejects missing/wrong weights, mapping, preprocessing or source/GPU.
No notebook has been executed on Kaggle in this implementation session.

Repository: https://github.com/NguyenDoKhaiHoan/DeepLearning
Exact clone/install commands: `notebooks/kaggle/README.md` in that repository.
The Git repository contains source only; attach the checkpoint bundle and video
as Kaggle Input separately. Restart the kernel if dependency installation changed
an already imported PyTorch/NumPy version.


In [ ]:
from pathlib import Path
import sys

# Point to the checked-out repository and real Kaggle Input bundle.
REPO_ROOT = Path("/kaggle/working/DeepLearning")
PROJECT_ROOT = REPO_ROOT / "traffic-mot"
BUNDLE_DIR = Path("/kaggle/input/REPLACE_ME/E1_p2_final")
UPSTREAM_REPO = REPO_ROOT / "RT-DETR/rtdetrv2_pytorch"
VIDEO = Path("/kaggle/input/REPLACE_ME/traffic.mp4")
OUTPUT_ROOT = Path("/kaggle/working/traffic_mot_results")
MOT_DATA_ROOT = Path("/kaggle/input/visdrone-mot")  # Use the actual Add Input mount path.
MOT_MANIFEST = OUTPUT_ROOT / "mot_split_manifest.json"  # Generated below for notebooks 02-04.
assert (PROJECT_ROOT / "src/traffic_mot").is_dir(), "Mount/clone this repository first"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from traffic_mot.utils.kaggle import (prepare_detection_config, check_video_dataset,
                                     check_mot_dataset, run_detection, run_stage)


In [ ]:
# VisDrone MOT uses image sequences; do not convert them into a fake-FPS MP4.
# Create from ZIPs or extracted directories; val is tuning, test-dev is held out.
from traffic_mot.utils.mot_data import prepare_mot_manifest
if not MOT_MANIFEST.exists():
    prepare_mot_manifest(MOT_MANIFEST, root=MOT_DATA_ROOT)
print("MOT_MANIFEST:", MOT_MANIFEST)
# Preparation lists frame/GT members without parsing held-out test annotations.
# This does not implement the missing tracker/evaluator/ablation CLI.


In [ ]:
config_path, preflight_report = prepare_detection_config(
    PROJECT_ROOT, BUNDLE_DIR, UPSTREAM_REPO, OUTPUT_ROOT / "configs",
    device="cuda:0", require_gpu=True)
print(preflight_report)


In [ ]:
print(check_mot_dataset(MOT_MANIFEST, split="test"))
FROZEN_CONFIG = OUTPUT_ROOT / "ablation/selected_config.yaml"
assert FROZEN_CONFIG.is_file(), "Finish validation ablations and freeze the selected configuration first"


In [ ]:
run_stage(PROJECT_ROOT, "final_evaluation.py", [
    "--config", str(FROZEN_CONFIG), "--detector-config", str(config_path),
    "--manifest", str(MOT_MANIFEST), "--split", "test",
    "--output-root", str(OUTPUT_ROOT / "final")])
# Planned: actual HOTA/IDF1/MOTA/IDSW, observed runtime components/FPS,
# recovery/size/occlusion tables and annotated demo clips. No placeholder scores.


## Acceptance evidence

Retain manifest, resolved config, source/checkpoint/data hashes, exact environment, raw predictions and decoded video verification. Only real runs on the declared split may populate metric tables. Missing assets/data/GPU are blockers, not zero scores.
